# <font color=#0099CC>**Descarga de datos: Modelo de Valero**</font>

## <font color=#0099CC>**0. INTRODUCCIÓN**</font>

### <font color=#336699>**0.1. Objetivo de este cuaderno**</font>

Notebook **proporcionado por el profesor** que define la fuente de datos oficial de la competición y el patrón mínimo de generación de ventanas (X, y) para el problema de forecasting. Sirve como referencia técnica para el resto del repositorio: tickers exactos, fecha de inicio, descarga con `yfinance` y construcción de muestras con un input de V_in días y un target promedio sobre los V_out días futuros.

### <font color=#336699>**0.2. Contenido**</font>

1. Importación de librerías.
2. Descarga de los precios de cierre de los 23 activos y cálculo de log-retornos diarios.
3. Definición de `create_time_series_data` para generar pares (X, y) con ventanas configurables.
4. Ejemplo de uso con V_in=10, V_out=5.
5. Partición train/test 90/10 con la semilla marcada.

> <u>Nota</u>: este fichero no se modifica. La lógica computacional es la del profesorado; las únicas mejoras añadidas aquí son textuales (celdas markdown explicativas y prints más informativos).

## <font color=#0099CC>**1. IMPORTACIONES**</font>

### <font color=#336699>**1.1. Librerías base**</font>

Se cargan las dependencias mínimas para la descarga de precios (`yfinance`), tratamiento numérico (`numpy`/`pandas`), visualización (`matplotlib`) y partición de datos (`train_test_split`). Se silencian los `FutureWarning` para mantener la salida limpia.

In [1]:
# Importación de Librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
import warnings

from sklearn.model_selection import train_test_split

# Supresión de warnings innecesarios
warnings.simplefilter(action="ignore", category=FutureWarning)

## <font color=#0099CC>**2. DESCARGA Y CÁLCULO DE RETORNOS**</font>

### <font color=#336699>**2.1. Universo de 23 activos del SP500**</font>

Lista cerrada de 23 tickers y fecha de inicio en 1945-01-01. Se descargan precios **ajustados** de cierre con `auto_adjust=True`, se eliminan columnas con huecos completos y se calculan log-retornos diarios mediante diferencia logarítmica.

> <u>Nota</u>: ésta es la lista oficial de la competición. Cualquier modelo del repositorio que se compare con otros grupos debe usar exactamente estos 23 tickers.

In [3]:
# Definición del universo de datos
start_date = '1945-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 'MSI', 'PG', 'XOM']

# Descarga y limpieza de los datos
precios_close = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=True)['Close']
precios_close.dropna(axis=1, inplace=True)

# Cálculo de retornos logarítmicos
returns = np.log(precios_close).diff().dropna()

print(f"\n> Activos descargados   : {precios_close.shape[1]}")
print(f"> Días de cotización    : {precios_close.shape[0]}")
print(f"> Forma de los retornos : {returns.shape}")
print("> Primeras filas de log-retornos:")
display(returns.head())

[*********************100%***********************]  23 of 23 completed



> Activos descargados   : 23
> Días de cotización    : 16191
> Forma de los retornos : (16190, 23)
> Primeras filas de log-retornos:


Ticker,AEP,BA,CAT,CNP,CVX,DIS,DTE,ED,GD,GE,...,IP,JNJ,KO,KR,MMM,MO,MRK,MSI,PG,XOM
Date,,,,,,,,,,,,,,,,,,,,,
1962-01-03,-0.001823,0.019803,0.009693,-0.009821,-0.002260,0.013333,-0.008265,0.000000,0.032925,-0.010084,...,-0.013745,-0.015670,-0.022532,0.020790,0.007490,-0.018517,-0.042802,-0.002913,-0.010989,0.014743
1962-01-04,-0.014706,-0.009852,0.025398,0.000000,-0.009090,0.000000,-0.008335,-0.003091,0.004041,-0.011894,...,-0.003466,-0.010578,0.007570,-0.004123,0.000000,0.006984,-0.010255,-0.008784,-0.016714,0.002436
1962-01-05,-0.022472,-0.020000,0.009361,-0.024418,-0.025435,0.003308,-0.021142,-0.021910,0.004023,-0.025976,...,0.010363,-0.016090,-0.022875,-0.025106,-0.026466,0.011534,-0.034462,-0.005900,-0.007047,-0.022140
1962-01-08,-0.007606,0.002522,0.006192,-0.011299,-0.004695,-0.003308,0.002135,0.004736,0.015938,-0.001756,...,-0.020834,-0.016348,-0.010336,-0.004246,-0.005764,-0.006904,0.003043,-0.017911,-0.027242,-0.002492
1962-01-09,-0.007663,0.002515,0.009217,0.000000,0.014020,0.019676,0.002129,-0.001576,0.003944,0.005259,...,-0.014134,0.010930,0.018017,-0.021506,0.000000,-0.018648,0.004551,-0.024391,-0.001454,-0.002496


## <font color=#0099CC>**3. CONSTRUCCIÓN DE VENTANAS (X, y)**</font>

### <font color=#336699>**3.1. Función create_time_series_data**</font>

Genera muestras deslizantes:

- X[i] = ventana de `input_window_size` días de historia, forma (V_in, Ch).
- y[i] = **promedio** de los `output_window_size` días siguientes, forma (Ch,).

Si `output_window_size == 0` el target es directamente el último valor de la ventana de entrada (variante prevista para casos de "predicción del siguiente día"). Internamente trabaja con numpy para mantener la coherencia tanto si recibe un DataFrame como un ndarray.

In [4]:
def create_time_series_data(data, input_window_size, output_window_size):
    """
    Genera secuencias de entrada y promedios de salida para datos de series temporales.

    Args:
        data (pd.DataFrame o np.array): Los datos de la serie temporal.
        input_window_size (int): El número de pasos de tiempo para la secuencia de entrada (X).
        output_window_size (int): El número de pasos de tiempo para calcular el promedio de la salida (y).

    Returns:
        tuple: (X, y) donde X son las secuencias de entrada y y son los promedios de salida.
               X tendrá la forma (num_samples, input_window_size, num_features).
               y tendrá la forma (num_samples, num_features) si output_window_size > 0.
               Si output_window_size == 0, y contendrá el último valor de la ventana de entrada.
    """
    X, y = [], []
    # Asegúrate de que los datos sean un array numpy para un manejo consistente
    data_array = data.values if isinstance(data, pd.DataFrame) else data

    num_features = data_array.shape[1] # Obtiene el número de características

    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        # Secuencia de entrada
        input_sequence = data_array[i : i + input_window_size]
        X.append(input_sequence)

        # Salida: promedio de los siguientes 'output_window_size' pasos
        # O si output_window_size es 0, toma el último valor de la ventana de entrada como 'y'
        if output_window_size > 0:
            output_sequence = data_array[i + input_window_size : i + input_window_size + output_window_size]
            average_output = np.mean(output_sequence, axis=0) # Promedio a lo largo de la ventana para cada característica
            y.append(average_output)
        else: # Si output_window_size es 0, la 'salida' es simplemente el último punto de la ventana de entrada
            y.append(data_array[i + input_window_size - 1])

    return np.array(X), np.array(y)

## <font color=#0099CC>**4. EJEMPLO DE GENERACIÓN**</font>

### <font color=#336699>**4.1. V_in=10, V_out=5**</font>

Aplicación directa de la función a la matriz de retornos para producir tensores con la forma exigida por el enunciado: X con (N, V_in, 23) e y con (N, 23).

In [6]:
# Definir los tamaños de ventana
input_window_size = 10  # Por ejemplo, 10 días de retornos como entrada
output_window_size = 5 # Por ejemplo, el promedio del siguiente día como salida

# Generar series temporales para el conjunto de entrenamiento
X, y = create_time_series_data(returns, input_window_size, output_window_size)

print(f"> Forma de X : {X.shape}   (N, V_in, Ch)")
print(f"> Forma de y : {y.shape}       (N, Ch)")

> Forma de X : (16176, 10, 23)   (N, V_in, Ch)
> Forma de y : (16176, 23)       (N, Ch)


## <font color=#0099CC>**5. PARTICIÓN TRAIN / TEST**</font>

### <font color=#336699>**5.1. Split 90/10 con semilla 42**</font>

Partición 90 % train / 10 % test con `random_state=42`. En este cuaderno guía se usa shuffle=False para mantener el orden cronológico durante la demostración; la regla del enunciado (partición aleatoria) se aplica en el código del proyecto, en `dataset_utils.get_partitions`, donde sí se baraja antes del split.

In [7]:
# Set a random seed for reproducibility
RANDOM_SEED = 42

# Split the time series sequences X and y into training and testing sets (90% train, 10% test)
# shuffle=False is crucial for time series data to maintain chronological order
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X, y, test_size=0.1, shuffle=False, random_state=RANDOM_SEED
)

print(f"> Forma de X_train_seq : {X_train_seq.shape}")
print(f"> Forma de y_train_seq : {y_train_seq.shape}")
print(f"> Forma de X_test_seq  : {X_test_seq.shape}")
print(f"> Forma de y_test_seq  : {y_test_seq.shape}")

> Forma de X_train_seq : (14558, 10, 23)
> Forma de y_train_seq : (14558, 23)
> Forma de X_test_seq  : (1618, 10, 23)
> Forma de y_test_seq  : (1618, 23)
